# MVP v0.3.3: Medium-Quality Behavior Data (Option B)

**Date:** 2026-03-13  
**Builds on:** v0.3.2.2 (guidance sweep)

## Motivation

v0.3.2.x failed because behavior data was expert-dominated (200 expert demos at 100% SR + 80 target rollouts).
This biases the diffuser's prior too high, leaving guidance no room to steer.

SOPE's reference implementation trains exclusively on **medium-quality behavior data** (~50% SR).
The medium anchor gives guidance bidirectional range: steer up for good policies, steer down for bad.

## Plan (Option B)

- **Behavior data**: 80 rollouts from `10demos_epoch20` (52% SR) — **no expert demos**
- **Stride 1** (not 2) to maximize chunks (~3600 from 80 rollouts)
- **2 target policies**: 200demos_epoch10 (18% SR) and 200demos_epoch40 (90% SR)
- **Normalization** from behavior data only (not mixed with expert)
- **BC behavior policy** trained on same medium-quality data

This is the cleanest match to SOPE's framework.

In [ ]:
import sys, os
import importlib
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm
from scipy import stats as sp_stats
import time

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

In [ ]:
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]
STATE_DIM = 19
ACTION_DIM = 7
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 26

# ── Behavior data: single medium-quality policy (Option B) ──
BEHAVIOR_POLICY_KEY = "10demos_epoch20"  # 52% SR — closest to SOPE's medium anchor
NUM_BEHAVIOR_ROLLOUTS = 80  # all available rollouts from this policy

# ── Chunk config ──
CHUNK_SIZE = 4
STRIDE = 1  # stride=1 to maximize chunks (SOPE uses stride=1)

# ── Diffusion config ──
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False  # x0-prediction

# ── Training config ──
TRAIN_EPOCHS = 50  # more epochs to compensate for less data
BATCH_SIZE = 64
LR = 3e-4
GRAD_CLIP = 1.0

# ── OPE config ──
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60

# ── Reward ──
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84

# ── Guidance config (SOPE defaults for diffusion policies) ──
K_GUIDE = 1
NORMALIZE_GRAD = True
USE_ADAPTIVE = False
CLAMP = False
L_INF = 1.0

# ── BC config ──
BC_EPOCHS = 500
BC_BATCH = 256
BC_HIDDEN = 256

# ── Guidance sweep ──
ACTION_SCALES = [0.0, 0.05, 0.1, 0.2, 0.5]
RATIOS = [0.0, 0.25, 0.5]

# Build sweep configs (deduplicate action_scale=0)
sweep_configs = []
for scale in ACTION_SCALES:
    if scale == 0.0:
        sweep_configs.append((0.0, 0.0))
    else:
        for ratio in RATIOS:
            sweep_configs.append((scale, ratio))

# ── Paths ──
CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"
ROLLOUT_BASE = PROJECT_ROOT / "rollouts"
BEHAVIOR_ROLLOUT_DIR = ROLLOUT_BASE / f"multi_policy_{BEHAVIOR_POLICY_KEY}"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-13"
DIFFUSION_CKPT_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v033_medium_behavior"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DIFFUSION_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── 2 target policies: one below, one above behavior anchor ──
POLICIES = {
    "200demos_epoch10": (CKPT_BASE / "lift_diffusion_200demos" / "20260311141036", 10),  # 18% SR
    "200demos_epoch40": (CKPT_BASE / "lift_diffusion_200demos" / "20260311141036", 40),  # 90% SR
}

# ── Oracle values (from oracle_eval_all_checkpoints.json, 50 rollouts each) ──
ORACLE_JSON = PROJECT_ROOT / "results/2026-03-12/oracle_eval_all_checkpoints.json"
with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)

oracle_values = {}
for key in POLICIES:
    od = oracle_data[key]
    oracle_values[key] = {
        "mean_return": od["mean_return"],
        "std_return": od["std_return"],
        "success_rate": od["success_rate"],
    }

sorted_policy_keys = sorted(POLICIES.keys(), key=lambda k: oracle_values[k]["mean_return"])

print(f"Behavior policy: {BEHAVIOR_POLICY_KEY} (SR={oracle_values[BEHAVIOR_POLICY_KEY]['success_rate']*100:.0f}%)" if BEHAVIOR_POLICY_KEY in oracle_values else f"Behavior policy: {BEHAVIOR_POLICY_KEY} (52% SR from oracle JSON)")
print(f"Behavior rollouts: {NUM_BEHAVIOR_ROLLOUTS} from {BEHAVIOR_ROLLOUT_DIR}")
print(f"Chunk config: size={CHUNK_SIZE}, stride={STRIDE}")
print(f"Training: {TRAIN_EPOCHS} epochs, batch={BATCH_SIZE}")
print(f"Sweep: {len(sweep_configs)} guidance configs x {len(POLICIES)} policies = {len(sweep_configs) * len(POLICIES)} evaluations")
print(f"\nTarget policies (sorted by oracle SR):")
for k in sorted_policy_keys:
    ov = oracle_values[k]
    print(f"  {k:>20s}: SR={ov['success_rate']*100:.0f}%")

## Step 1: Load Behavior Data (Medium-Quality Only)

In [ ]:
behavior_episodes = []
all_states_list = []
all_actions_list = []

rollout_paths = sorted(BEHAVIOR_ROLLOUT_DIR.glob("rollout_*.h5"))[:NUM_BEHAVIOR_ROLLOUTS]
assert len(rollout_paths) >= NUM_BEHAVIOR_ROLLOUTS, \
    f"Expected {NUM_BEHAVIOR_ROLLOUTS} rollouts, found {len(rollout_paths)}"

for path in tqdm(rollout_paths, desc="Loading behavior rollouts"):
    with h5py.File(path, "r") as f:
        latents = f["latents"][:]
        actions = f["actions"][:]
        rewards = f["rewards"][:] if "rewards" in f else np.zeros(len(actions))
    
    states = latents[:, -1, :] if latents.ndim == 3 else latents
    states = states.astype(np.float32)
    actions = actions.astype(np.float32)
    
    episode = {
        "states": states[:-1],
        "actions": actions[:-1],
        "rewards": rewards[:-1] if len(rewards) > 1 else rewards,
        "next-states": states[1:],
    }
    behavior_episodes.append(episode)
    all_states_list.append(states)
    all_actions_list.append(actions)

# Compute stats
all_states_cat = np.concatenate(all_states_list, axis=0)
all_actions_cat = np.concatenate(all_actions_list, axis=0)
total_transitions = sum(len(ep["states"]) for ep in behavior_episodes)

# Success rate from env reward (note: obs recording bug means cube_z-based SR will be 0%)
env_successes = [ep["rewards"].sum() > 0 for ep in behavior_episodes]
env_sr = np.mean(env_successes)
cube_z_successes = [ep["states"][:, CUBE_Z_INDEX].max() > LIFT_THRESHOLD for ep in behavior_episodes]
cube_z_sr = np.mean(cube_z_successes)

# Behavior policy oracle (from full oracle JSON, not just target policies)
behavior_oracle = oracle_data[BEHAVIOR_POLICY_KEY]

print(f"\nLoaded {len(behavior_episodes)} behavior episodes ({BEHAVIOR_POLICY_KEY})")
print(f"Total transitions: {total_transitions}")
print(f"State dim: {all_states_cat.shape[1]}, Action dim: {all_actions_cat.shape[1]}")
print(f"Env reward SR: {env_sr*100:.1f}% (from rollout rewards)")
print(f"Cube_z SR: {cube_z_sr*100:.1f}% (from obs — affected by recording bug)")
print(f"Oracle SR: {behavior_oracle['success_rate']*100:.0f}%")
print(f"\nNO expert demos in training data — medium-quality only (matching SOPE assumption)")

## Step 2: Chunk Behavior Data + Compute Normalization

In [ ]:
# Normalization from behavior data only
norm_mean = np.concatenate([all_states_cat.mean(0), all_actions_cat.mean(0)]).astype(np.float32)
norm_std = np.maximum(
    np.concatenate([all_states_cat.std(0), all_actions_cat.std(0)]), 1e-6
).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x, m=norm_mean_t, s=norm_std_t: (x - m) / s
unnormalize_fn = lambda x, m=norm_mean_t, s=norm_std_t: x * s + m

# Chunk with stride=1
chunks_s, chunks_a = [], []
for ep in behavior_episodes:
    st, act = ep["states"], ep["actions"]
    T = len(st)
    if T < CHUNK_SIZE + 1:
        continue
    for t0 in range(0, T - CHUNK_SIZE, STRIDE):
        chunks_s.append(st[t0:t0 + CHUNK_SIZE + 1])  # +1 for conditioning state
        chunks_a.append(act[t0:t0 + CHUNK_SIZE])

chunks_s = np.array(chunks_s, dtype=np.float32)
chunks_a = np.array(chunks_a, dtype=np.float32)

# Training tensors: x = [states, actions] for chunk, cond = first state
train_x = np.concatenate([chunks_s[:, :-1, :], chunks_a], axis=-1)  # (N, chunk_size, 26)
train_cond = chunks_s[:, 0, :]  # (N, 19)
train_x_t = torch.tensor(train_x, dtype=torch.float32, device=device)
train_cond_t = torch.tensor(train_cond, dtype=torch.float32, device=device)

print(f"Chunked {len(behavior_episodes)} episodes into {len(train_x_t)} training chunks")
print(f"  chunk_size={CHUNK_SIZE}, stride={STRIDE}")
print(f"  train_x shape: {train_x_t.shape}")
print(f"  train_cond shape: {train_cond_t.shape}")
print(f"  Batches per epoch: {len(train_x_t) // BATCH_SIZE}")
print(f"  Normalization computed from behavior data only (no expert demos)")

## Step 3: Train Chunk Diffuser on Behavior Data

In [ ]:
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
    dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn, unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON, loss_type="l2",
    clip_denoised=False, action_weight=ACTION_WEIGHT,
    loss_weights=None, loss_discount=1.0,
).to(device)

ema = EMA(diffusion_model)
optimizer = torch.optim.Adam(diffusion_model.parameters(), lr=LR)

n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Model: {n_params:,} parameters")
print(f"Training for {TRAIN_EPOCHS} epochs on {len(train_x_t)} chunks...")

t0_train = time.time()
diffusion_model.train()
epoch_losses = []

for ep_num in range(1, TRAIN_EPOCHS + 1):
    perm = torch.randperm(len(train_x_t), device=device)
    ep_loss = []
    for start in range(0, len(train_x_t) - BATCH_SIZE + 1, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        x_norm = normalize_fn(train_x_t[idx])
        c_norm = normalize_fn(
            torch.cat([train_cond_t[idx], torch.zeros(BATCH_SIZE, ACTION_DIM, device=device)], dim=-1)
        )[:, :STATE_DIM]
        loss, _ = diffusion_model.loss(x_norm, {0: c_norm})
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), GRAD_CLIP)
        optimizer.step()
        ema.update(diffusion_model)
        ep_loss.append(loss.item())
    epoch_losses.append(np.mean(ep_loss))
    if ep_num % 5 == 0 or ep_num == 1:
        print(f"  Epoch {ep_num:>3d}/{TRAIN_EPOCHS}: loss={epoch_losses[-1]:.6f}")

elapsed_train = time.time() - t0_train
print(f"\nDiffuser trained: {elapsed_train:.0f}s, final loss={epoch_losses[-1]:.6f}")

# Save checkpoint
torch.save(diffusion_model.state_dict(), DIFFUSION_CKPT_DIR / "diffusion_model.pt")
torch.save(ema.ema_model.state_dict(), DIFFUSION_CKPT_DIR / "diffusion_model_ema.pt")
np.save(DIFFUSION_CKPT_DIR / "norm_mean.npy", norm_mean)
np.save(DIFFUSION_CKPT_DIR / "norm_std.npy", norm_std)
print(f"Checkpoint saved to {DIFFUSION_CKPT_DIR}")

In [ ]:
# Training loss curve
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o', markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(f"Chunk Diffuser Training ({TRAIN_EPOCHS} epochs, {len(train_x_t)} chunks, behavior data only)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3b: Chunk Reconstruction Sanity Check

In [ ]:
# Denoise a batch of real chunks to check reconstruction quality
ema.ema_model.eval()
diffusion_model.eval()

n_recon = min(64, len(train_x_t))
recon_idx = torch.randperm(len(train_x_t))[:n_recon]
real_chunks = train_x_t[recon_idx]
real_cond = train_cond_t[recon_idx]

# Normalize
real_norm = normalize_fn(real_chunks)
cond_norm = normalize_fn(
    torch.cat([real_cond, torch.zeros(n_recon, ACTION_DIM, device=device)], dim=-1)
)[:, :STATE_DIM]

# Generate from noise with same conditioning
with torch.no_grad():
    shape = real_norm.shape
    x = torch.randn(shape, device=device)
    conditions = {0: cond_norm}
    x = apply_conditioning(x, conditions, STATE_DIM)
    
    for t_diff in reversed(range(N_DIFFUSION_STEPS)):
        t_tensor = torch.full((n_recon,), t_diff, device=device, dtype=torch.long)
        model_mean, _, model_log_variance = ema.ema_model.p_mean_variance(x=x, t=t_tensor)
        model_std = torch.exp(0.5 * model_log_variance)
        noise = torch.randn_like(x)
        nonzero_mask = (1 - (t_diff == 0) * 1.0)
        x = model_mean + nonzero_mask * model_std * noise
        x = apply_conditioning(x, conditions, STATE_DIM)

recon_chunks = unnormalize_fn(x).cpu().numpy()
real_chunks_np = real_chunks.cpu().numpy()

# L2 error
l2_states = np.sqrt(((recon_chunks[:, :, :STATE_DIM] - real_chunks_np[:, :, :STATE_DIM]) ** 2).mean())
l2_actions = np.sqrt(((recon_chunks[:, :, STATE_DIM:] - real_chunks_np[:, :, STATE_DIM:]) ** 2).mean())
print(f"Reconstruction RMSE — states: {l2_states:.6f}, actions: {l2_actions:.6f}")

# Visualize
KEY_DIMS = {"cube_z": 2, "eef_z": 12, "cube_x": 0, "eef_x": 10, "gripper_q0": 17}
fig, axes = plt.subplots(1, len(KEY_DIMS), figsize=(4 * len(KEY_DIMS), 3))
for idx, (name, dim) in enumerate(KEY_DIMS.items()):
    ax = axes[idx]
    for j in range(min(5, n_recon)):
        ax.plot(real_chunks_np[j, :, dim], color="green", alpha=0.4,
                label="Real" if j == 0 else "")
        ax.plot(recon_chunks[j, :, dim], color="coral", alpha=0.4, linestyle="--",
                label="Reconstructed" if j == 0 else "")
    ax.set_title(name, fontsize=10)
    ax.grid(True, alpha=0.2)
    if idx == 0:
        ax.legend(fontsize=7)

plt.suptitle("Chunk Reconstruction: Real vs Denoised", fontweight="bold")
plt.tight_layout()
plt.show()

## Step 4: Train BC Behavior Policy

In [ ]:
class BCGaussian(nn.Module):
    """Simple BC policy with Gaussian output for SOPE-style guidance."""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, state):
        h = self.net(state)
        return self.mean_head(h), self.log_std_head(h).clamp(-5, 2)

    def log_prob(self, state, action):
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        return -0.5 * (((action - mean) / std) ** 2 + 2 * log_std + math.log(2 * math.pi)).sum(dim=-1)

    def grad_log_prob(self, state, action):
        with torch.no_grad():
            mean, log_std = self.forward(state)
            std = torch.exp(log_std)
            return -(action - mean) / (std ** 2)

    def grad_log_prob_chunk(self, states, actions):
        B, T, _ = states.shape
        return self.grad_log_prob(
            states.reshape(B * T, -1), actions.reshape(B * T, -1)
        ).reshape(B, T, -1)


# Train on behavior data (medium-quality rollouts only)
bc_states = np.concatenate([ep["states"] for ep in behavior_episodes], axis=0)
bc_actions = np.concatenate([ep["actions"] for ep in behavior_episodes], axis=0)
print(f"BC training data: {len(bc_states)} (state, action) pairs from {BEHAVIOR_POLICY_KEY}")

bc_behavior = BCGaussian(STATE_DIM, ACTION_DIM, hidden_dim=BC_HIDDEN).to(device)
bc_optimizer = torch.optim.Adam(bc_behavior.parameters(), lr=1e-3)
states_t = torch.tensor(bc_states, dtype=torch.float32, device=device)
actions_t = torch.tensor(bc_actions, dtype=torch.float32, device=device)

bc_behavior.train()
bc_losses = []
for bc_ep in range(BC_EPOCHS):
    idx = torch.randint(0, len(states_t), (BC_BATCH,), device=device)
    nll = -bc_behavior.log_prob(states_t[idx], actions_t[idx]).mean()
    bc_optimizer.zero_grad()
    nll.backward()
    bc_optimizer.step()
    bc_losses.append(nll.item())

bc_behavior.eval()
print(f"BC trained: final NLL={bc_losses[-1]:.4f}")

# Sanity check
test_grad = bc_behavior.grad_log_prob(states_t[:8], actions_t[:8])
print(f"Behavior grad norms: {test_grad.norm(dim=-1).cpu().numpy()}")

## Step 5: Helper Functions

In [ ]:
def get_initial_states_from_data(data, num_samples, device):
    """Sample initial states from a list of episodes."""
    all_initial = np.array([ep["states"][0] for ep in data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def score_trajectories_gt(trajectories, cube_z_index, threshold):
    """Episode-level binary reward: 1.0 if cube_z > threshold at ANY timestep."""
    cube_z = trajectories[:, :, cube_z_index]
    successes = np.any(cube_z > threshold, axis=1)
    return successes.astype(np.float32), successes


def generate_trajectories_full_guidance(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim,
    chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    action_scale=0.0, ratio=0.5,
    k_guide=1, normalize_grad=True,
    use_adaptive=False, clamp=False, l_inf=1.0,
):
    """Generate trajectories with full SOPE-style guidance."""
    guided = (target_scorer is not None and action_scale > 0)
    use_neg_grad = (behavior_scorer is not None and ratio > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    n_timesteps = diffusion_model.n_timesteps

    padded = torch.cat([initial_states, torch.zeros(batch_size, action_dim, device=device)], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]
    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0

    while total_generated < t_gen:
        shape = (batch_size, chunk_size, transition_dim)
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)
            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)

            if guided:
                model_mean = unnormalize_fn(model_mean)
                for _k in range(k_guide):
                    states_chunk = model_mean[:, :, :state_dim]
                    actions_chunk = model_mean[:, :, state_dim:]
                    target_grad = target_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    if use_neg_grad:
                        behavior_grad = behavior_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    if normalize_grad:
                        eps = 1e-6
                        target_grad = target_grad / (target_grad.norm(dim=-1, keepdim=True) + eps)
                        if use_neg_grad:
                            behavior_grad = behavior_grad / (behavior_grad.norm(dim=-1, keepdim=True) + eps)
                    if clamp:
                        target_grad = target_grad.clamp(-l_inf, l_inf)
                        if use_neg_grad:
                            behavior_grad = behavior_grad.clamp(-l_inf, l_inf)
                    if use_neg_grad:
                        guide_actions = target_grad - ratio * behavior_grad
                    else:
                        guide_actions = target_grad
                    guide = torch.zeros_like(model_mean)
                    guide[:, :, state_dim:] = guide_actions
                    if use_adaptive:
                        scale_mult = 0.5 * (1 + math.cos(math.pi * (n_timesteps - t_diff) / n_timesteps))
                        guide = scale_mult * action_scale * guide
                    else:
                        guide = action_scale * guide
                    model_mean = model_mean + guide
                    model_mean = normalize_fn(model_mean)
                    model_mean = apply_conditioning(model_mean, conditions, state_dim)
                    model_mean = unnormalize_fn(model_mean)
                model_mean = normalize_fn(model_mean)

            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_unnorm = unnormalize_fn(x)
        steps_to_store = min(chunk_size - 1, t_gen - total_generated)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store
        if total_generated >= t_gen:
            break
        conditions = {0: x[:, -1, :state_dim]}

    return all_trajectories.detach().cpu().numpy()


print("Helper functions defined.")

## Step 6: Load Target Policy Scorers

In [ ]:
target_scorers = {}
for policy_key, (run_dir, epoch) in POLICIES.items():
    ckpt = load_checkpoint(str(run_dir), ckpt_path=f"models/model_epoch_{epoch}.pth")
    target_algo = build_algo_from_checkpoint(ckpt, device=str(device))
    target_scorers[policy_key] = RobomimicDiffusionScorer(target_algo, device=str(device), obs_keys=OBS_KEYS)
    print(f"  {policy_key}: loaded")

print(f"\nLoaded {len(target_scorers)} target policy scorers")

# Compare score magnitudes
test_s = states_t[:8]
test_a = actions_t[:8]
print(f"\nScore magnitude comparison (8 test states):")
behavior_norms = bc_behavior.grad_log_prob(test_s, test_a).norm(dim=-1).mean().item()
print(f"  Behavior (BC): {behavior_norms:.1f}")
for pk in sorted_policy_keys:
    target_norms = target_scorers[pk].grad_log_prob(test_s, test_a).norm(dim=-1).mean().item()
    print(f"  {pk}: {target_norms:.1f}")

## Step 7: Load Target Policy Rollouts (for initial states)

In [ ]:
# Load rollouts for each target policy (for sampling initial states)
target_data_by_policy = {}
for policy_key in POLICIES:
    rollout_dir = ROLLOUT_BASE / f"multi_policy_{policy_key}"
    existing = sorted(rollout_dir.glob("rollout_*.h5"))
    policy_episodes = []
    for path in existing[:20]:  # 20 rollouts for initial state sampling
        with h5py.File(path, "r") as f:
            latents = f["latents"][:]
            actions = f["actions"][:]
            rewards = f["rewards"][:] if "rewards" in f else np.zeros(len(actions))
        states = latents[:, -1, :] if latents.ndim == 3 else latents
        states = states.astype(np.float32)
        actions = actions.astype(np.float32)
        episode = {
            "states": states[:-1], "actions": actions[:-1],
            "rewards": rewards[:-1] if len(rewards) > 1 else rewards,
            "next-states": states[1:],
        }
        policy_episodes.append(episode)
    target_data_by_policy[policy_key] = policy_episodes
    env_sr = np.mean([ep["rewards"].sum() > 0 for ep in policy_episodes])
    print(f"  {policy_key:>20s}: {len(policy_episodes)} rollouts loaded, env SR={env_sr*100:.0f}%")

print(f"\nInitial states will be sampled from each target policy's own rollouts.")

## Step 8: Guidance Sweep

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

sweep_results = {}
t0_sweep = time.time()
total_evals = len(sweep_configs) * len(POLICIES)
eval_count = 0

for cfg_idx, (scale, ratio) in enumerate(sweep_configs):
    cfg_key = (scale, ratio)
    sweep_results[cfg_key] = {}

    for policy_key in POLICIES:
        eval_count += 1
        oracle_v = oracle_values[policy_key]["mean_return"]

        # Deterministic seed per policy (same initial states across configs)
        np.random.seed(42 + list(POLICIES.keys()).index(policy_key))
        torch.manual_seed(42 + list(POLICIES.keys()).index(policy_key))

        initial_states = get_initial_states_from_data(
            target_data_by_policy[policy_key], NUM_SYNTHETIC_TRAJS, device
        )

        trajs = generate_trajectories_full_guidance(
            diffusion_model=ema.ema_model,
            initial_states=initial_states,
            normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
            state_dim=STATE_DIM, action_dim=ACTION_DIM,
            chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
            target_scorer=target_scorers[policy_key] if scale > 0 else None,
            behavior_scorer=bc_behavior if (scale > 0 and ratio > 0) else None,
            action_scale=scale, ratio=ratio,
            k_guide=K_GUIDE, normalize_grad=NORMALIZE_GRAD,
            use_adaptive=USE_ADAPTIVE, clamp=CLAMP, l_inf=L_INF,
        )

        returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD)
        ope = float(np.mean(returns))
        sr = float(np.mean(successes))

        sweep_results[cfg_key][policy_key] = {
            "ope": ope, "sr": sr, "oracle": oracle_v,
            "rel_error": abs(ope - oracle_v) / max(abs(oracle_v), 1e-6),
            "trajs": trajs,  # keep for visualization
        }

    # Print config summary
    opes = [sweep_results[cfg_key][k]["ope"] for k in sorted_policy_keys]
    oracles = [sweep_results[cfg_key][k]["oracle"] for k in sorted_policy_keys]
    if len(set(opes)) > 1:
        rho, _ = sp_stats.spearmanr(oracles, opes)
    else:
        rho = float('nan')
    srs = [sweep_results[cfg_key][k]["sr"] for k in sorted_policy_keys]
    mae = np.mean(np.abs(np.array(oracles) - np.array(opes)))
    print(f"[{eval_count}/{total_evals}] s={scale}, r={ratio}: "
          f"SRs={[f'{s*100:.0f}%' for s in srs]}, rho={rho:.3f}, MAE={mae:.3f}")

elapsed_sweep = time.time() - t0_sweep
print(f"\nSweep complete: {elapsed_sweep/60:.1f} min")

## Step 9: Results Summary Table

In [ ]:
print(f"{'scale':>6s} {'ratio':>6s} | ", end="")
print(" | ".join([f"{k[:15]:>15s}" for k in sorted_policy_keys]), end="")
print(f" | {'rho':>6s} {'MAE':>6s}")
print("-" * 160)

best_cfg = None
best_rho = -2
best_mae = 999

for (scale, ratio), results in sweep_results.items():
    oracles = [results[k]["oracle"] for k in sorted_policy_keys]
    opes = [results[k]["ope"] for k in sorted_policy_keys]
    srs = [results[k]["sr"] for k in sorted_policy_keys]

    if len(set(opes)) > 1:
        rho, _ = sp_stats.spearmanr(oracles, opes)
    else:
        rho = float('nan')
    mae = np.mean(np.abs(np.array(oracles) - np.array(opes)))

    row = f"{scale:>6.2f} {ratio:>6.2f} | "
    row += " | ".join([f"{ope:.2f} ({sr*100:3.0f}%)" for ope, sr in zip(opes, srs)])
    row += f" | {rho:>6.3f} {mae:>6.3f}"
    print(row)

    if not np.isnan(rho) and (rho > best_rho or (rho == best_rho and mae < best_mae)):
        best_rho = rho
        best_mae = mae
        best_cfg = (scale, ratio)

# Oracle row
print()
row = f"{'ORACLE':>13s} | "
row += " | ".join([f"{oracle_values[k]['mean_return']:.2f} ({oracle_values[k]['success_rate']*100:3.0f}%)" for k in sorted_policy_keys])
print(row)

print(f"\nBest config: scale={best_cfg[0]}, ratio={best_cfg[1]} (rho={best_rho:.3f}, MAE={best_mae:.3f})")

## Step 10: Visualizations

In [ ]:
# ── Figure 1: Heatmap of Spearman rho and MAE across sweep grid ──
unique_scales = sorted(set(s for s, r in sweep_configs if s > 0))
unique_ratios = sorted(set(r for s, r in sweep_configs if s > 0))

rho_matrix = np.full((len(unique_scales), len(unique_ratios)), np.nan)
mae_matrix = np.full((len(unique_scales), len(unique_ratios)), np.nan)

for i, scale in enumerate(unique_scales):
    for j, ratio in enumerate(unique_ratios):
        cfg_key = (scale, ratio)
        if cfg_key in sweep_results:
            oracles = [sweep_results[cfg_key][k]["oracle"] for k in sorted_policy_keys]
            opes = [sweep_results[cfg_key][k]["ope"] for k in sorted_policy_keys]
            if len(set(opes)) > 1:
                rho_matrix[i, j], _ = sp_stats.spearmanr(oracles, opes)
            mae_matrix[i, j] = np.mean(np.abs(np.array(oracles) - np.array(opes)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(rho_matrix, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(unique_ratios)))
ax.set_xticklabels([f"{r}" for r in unique_ratios])
ax.set_yticks(range(len(unique_scales)))
ax.set_yticklabels([f"{s}" for s in unique_scales])
ax.set_xlabel("ratio (negative guidance)")
ax.set_ylabel("action_scale")
ax.set_title("Spearman rho (higher = better ranking)")
for i in range(len(unique_scales)):
    for j in range(len(unique_ratios)):
        val = rho_matrix[i, j]
        txt = f"{val:.2f}" if not np.isnan(val) else "NaN"
        ax.text(j, i, txt, ha="center", va="center", fontsize=9, fontweight="bold")
fig.colorbar(im, ax=ax)

ax = axes[1]
im = ax.imshow(mae_matrix, cmap="RdYlGn_r", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(len(unique_ratios)))
ax.set_xticklabels([f"{r}" for r in unique_ratios])
ax.set_yticks(range(len(unique_scales)))
ax.set_yticklabels([f"{s}" for s in unique_scales])
ax.set_xlabel("ratio (negative guidance)")
ax.set_ylabel("action_scale")
ax.set_title("MAE (lower = better calibration)")
for i in range(len(unique_scales)):
    for j in range(len(unique_ratios)):
        val = mae_matrix[i, j]
        txt = f"{val:.3f}" if not np.isnan(val) else "NaN"
        ax.text(j, i, txt, ha="center", va="center", fontsize=9, fontweight="bold")
fig.colorbar(im, ax=ax)

plt.suptitle("v0.3.3: Guidance Sweep — Medium Behavior Data (Option B)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Oracle vs OPE scatter for best config ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Best guided config
ax = axes[0]
for k in sorted_policy_keys:
    r = sweep_results[best_cfg][k]
    ax.scatter(r["oracle"], r["ope"], s=80, zorder=5)
    ax.annotate(k, (r["oracle"], r["ope"]), textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="ideal")
ax.set_xlabel("Oracle V^pi")
ax.set_ylabel("OPE Estimate")
ax.set_title(f"Best: s={best_cfg[0]}, r={best_cfg[1]} (rho={best_rho:.3f})")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

# Unguided baseline
ax = axes[1]
unguided = sweep_results[(0.0, 0.0)]
opes_ug = [unguided[k]["ope"] for k in sorted_policy_keys]
oracles_ug = [unguided[k]["oracle"] for k in sorted_policy_keys]
if len(set(opes_ug)) > 1:
    rho_ug, _ = sp_stats.spearmanr(oracles_ug, opes_ug)
else:
    rho_ug = float('nan')
for k in sorted_policy_keys:
    r = unguided[k]
    ax.scatter(r["oracle"], r["ope"], s=80, zorder=5)
    ax.annotate(k, (r["oracle"], r["ope"]), textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="ideal")
ax.set_xlabel("Oracle V^pi")
ax.set_ylabel("OPE Estimate")
ax.set_title(f"Unguided baseline (rho={rho_ug:.3f})")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("v0.3.3: Oracle vs OPE — Medium Behavior Data", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Per-policy SR across all configs ──
fig, axes = plt.subplots(1, len(sorted_policy_keys), figsize=(4 * len(sorted_policy_keys), 4), sharey=True)

for ax, policy_key in zip(axes, sorted_policy_keys):
    oracle_sr = oracle_values[policy_key]["success_rate"]
    config_labels = []
    config_srs = []
    for (scale, ratio) in sweep_configs:
        label = f"s{scale}_r{ratio}"
        sr = sweep_results[(scale, ratio)][policy_key]["sr"]
        config_labels.append(label)
        config_srs.append(sr * 100)

    colors = ["steelblue" if s == 0 else ("orange" if r == 0 else "coral") for s, r in sweep_configs]
    ax.bar(range(len(config_labels)), config_srs, color=colors)
    ax.axhline(y=oracle_sr * 100, color="green", linestyle="--", linewidth=1.5,
               label=f"Oracle={oracle_sr*100:.0f}%")
    ax.set_xticks(range(len(config_labels)))
    ax.set_xticklabels(config_labels, rotation=70, ha="right", fontsize=6)
    ax.set_title(f"{policy_key}\n(Oracle={oracle_sr*100:.0f}%)", fontsize=8)
    ax.set_ylim(0, 105)
    if ax == axes[0]:
        ax.set_ylabel("Synthetic SR (%)")
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.2, axis="y")

plt.suptitle("v0.3.3: Synthetic SR by Config — Medium Behavior Data", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 4: Cube z trajectories for unguided and best config ──
show_policies = sorted_policy_keys  # both policies

fig, axes = plt.subplots(2, len(show_policies), figsize=(6 * len(show_policies), 8))

for col, pk in enumerate(show_policies):
    oracle_sr = oracle_values[pk]["success_rate"]

    # Unguided
    ax = axes[0, col]
    trajs_ug = sweep_results[(0.0, 0.0)][pk]["trajs"]
    for j in range(min(15, NUM_SYNTHETIC_TRAJS)):
        ax.plot(trajs_ug[j, :, CUBE_Z_INDEX], color="steelblue", alpha=0.2)
    ax.plot(trajs_ug[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkblue", linewidth=2, label="Mean")
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(f"{pk}\nUnguided (Oracle={oracle_sr*100:.0f}%)", fontsize=9)
    ax.set_ylim([0.78, 0.95])
    ax.set_ylabel("cube_z")
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=7)

    # Best guided
    ax = axes[1, col]
    trajs_best = sweep_results[best_cfg][pk]["trajs"]
    for j in range(min(15, NUM_SYNTHETIC_TRAJS)):
        ax.plot(trajs_best[j, :, CUBE_Z_INDEX], color="coral", alpha=0.2)
    ax.plot(trajs_best[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkred", linewidth=2, label="Mean")
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    sr_best = sweep_results[best_cfg][pk]["sr"]
    ax.set_title(f"Best guided s={best_cfg[0]},r={best_cfg[1]}\nSR={sr_best*100:.0f}%", fontsize=9)
    ax.set_ylim([0.78, 0.95])
    ax.set_xlabel("Timestep")
    ax.set_ylabel("cube_z")
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=7)

plt.suptitle("v0.3.3: Cube Z — Unguided (top) vs Best Guided (bottom)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 5: Per-dimension trajectory comparison for best config (2 policies x 6 dims) ──
KEY_STATE_DIMS = {
    "cube_x": 0, "cube_y": 1, "cube_z": 2,
    "eef_x": 10, "eef_y": 11, "eef_z": 12,
}

fig, axes = plt.subplots(len(show_policies), len(KEY_STATE_DIMS),
                         figsize=(3.5 * len(KEY_STATE_DIMS), 3.5 * len(show_policies)))

for row, pk in enumerate(show_policies):
    trajs_ug = sweep_results[(0.0, 0.0)][pk]["trajs"]
    trajs_best = sweep_results[best_cfg][pk]["trajs"]

    for col, (dim_name, dim_idx) in enumerate(KEY_STATE_DIMS.items()):
        ax = axes[row, col]
        for j in range(min(8, NUM_SYNTHETIC_TRAJS)):
            ax.plot(trajs_ug[j, :, dim_idx], color="steelblue", alpha=0.15,
                    label="Unguided" if j == 0 and col == 0 else "")
            ax.plot(trajs_best[j, :, dim_idx], color="coral", alpha=0.15,
                    label="Best guided" if j == 0 and col == 0 else "")
        if dim_idx == CUBE_Z_INDEX:
            ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.4)
        if row == 0:
            ax.set_title(dim_name, fontsize=10, fontweight="bold")
        if col == 0:
            oracle_sr = oracle_values[pk]["success_rate"]
            ax.set_ylabel(f"{pk}\n(Oracle={oracle_sr*100:.0f}%)", fontsize=8)
        ax.grid(True, alpha=0.2)
        ax.tick_params(labelsize=7)
        if row == 0 and col == 0:
            ax.legend(fontsize=6)

plt.suptitle("v0.3.3: Per-Dimension Trajectories — Unguided (blue) vs Best Guided (red)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 11: Summary

In [ ]:
# Clean up trajectory data from sweep_results to free memory
for cfg_key in sweep_results:
    for pk in sweep_results[cfg_key]:
        sweep_results[cfg_key][pk].pop("trajs", None)

del target_scorers
torch.cuda.empty_cache()

print(f"{'='*70}")
print(f"MVP v0.3.3 COMPLETE — Medium Behavior Data (Option B)")
print(f"{'='*70}")
print(f"Behavior data: {len(behavior_episodes)} episodes from {BEHAVIOR_POLICY_KEY} (52% SR)")
print(f"Training chunks: {len(train_x_t)} (stride={STRIDE})")
print(f"Training: {TRAIN_EPOCHS} epochs, final loss={epoch_losses[-1]:.6f}")
print(f"Diffuser training time: {elapsed_train:.0f}s")
print(f"Sweep: {len(sweep_configs)} configs x {len(POLICIES)} policies, {elapsed_sweep/60:.1f} min")
print(f"\nBest config: scale={best_cfg[0]}, ratio={best_cfg[1]}")
print(f"  Spearman rho: {best_rho:.3f}")
print(f"  MAE: {best_mae:.3f}")
print(f"\nPer-policy results (best config):")
for k in sorted_policy_keys:
    r = sweep_results[best_cfg][k]
    print(f"  {k:>20s}: Oracle={r['oracle']:.2f}, OPE={r['ope']:.2f}, SR={r['sr']*100:.0f}%")

print(f"\nUnguided baseline:")
ug = sweep_results[(0.0, 0.0)]
opes_ug = [ug[k]["ope"] for k in sorted_policy_keys]
oracles_ug = [ug[k]["oracle"] for k in sorted_policy_keys]
if len(set(opes_ug)) > 1:
    rho_ug, _ = sp_stats.spearmanr(oracles_ug, opes_ug)
else:
    rho_ug = float('nan')
mae_ug = np.mean(np.abs(np.array(oracles_ug) - np.array(opes_ug)))
print(f"  rho={rho_ug:.3f}, MAE={mae_ug:.3f}")
for k in sorted_policy_keys:
    r = ug[k]
    print(f"  {k:>20s}: Oracle={r['oracle']:.2f}, OPE={r['ope']:.2f}, SR={r['sr']*100:.0f}%")

print(f"\nKey question: Does guidance improve ranking (rho) over unguided?")
print(f"  Unguided rho: {rho_ug:.3f}")
print(f"  Best guided rho: {best_rho:.3f}")
print(f"  Improvement: {best_rho - rho_ug:+.3f}")